In [2]:
import os
import math
import matplotlib.pyplot as plt

import verde as vd
import numpy as np
import pandas as pd
import geopandas as gpd



In [3]:
_dir = '/home/jan/Documents/EPFL/project_2/data/labels/dataset_v4'

dfs = []
#sample_dst = []
for _file in os.listdir(_dir):
    if not _file.endswith('.csv'):
        continue
    if _file in ['labels.csv', 'cocoa_th_65.csv']:
        continue
    print(_file)
    df = pd.read_csv(os.path.join(_dir, _file))    
    df['dataset'] = _file
    
    
    '''try:
        dst_samples = df.sample(100, random_state=1)
    except ValueError:
        dst_samples = df
        print(dst_samples.shape)
    dst_samples['dataset'] = _file
    sample_dst.append(dst_samples)'''

    dfs.append(df)


labels_55k.csv
mines.csv


In [93]:
df = pd.concat(dfs)

In [94]:
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])


In [95]:
gdf = gpd.GeoDataFrame(df)
gdf['geometry'] = gdf.geometry.centroid

In [96]:
gdf = gdf.reset_index(drop=True)

In [97]:
cls, count = np.unique(gdf.driver, return_counts=True)


for i in range(len(cls)):
    print(cls[i], count[i])

aquaculture 33
beach, dune, snad 496
citrus 40
cocoa 14923
coffee 431
cotton 37
forest plantation 58518
forest regrowth 46390
hypersaline tidal flat 129
mining 38842
other temporary crops 3202
palm oil 368508
pasture 563567
rice 40581
river, lake, ocean 9999
rocky outcrop 701
rubber 48150
shifting agriculture 19806
soy 50382
sugar cane 1398
urban 22281
water 3


In [98]:
gdf.driver.unique()

array(['rubber', 'cocoa', 'river, lake, ocean', 'pasture',
       'other temporary crops', 'mining', 'soy', 'urban', 'palm oil',
       'aquaculture', 'sugar cane', 'forest plantation',
       'beach, dune, snad', 'rocky outcrop', 'hypersaline tidal flat',
       'rice', 'cotton', 'coffee', 'citrus', 'forest regrowth',
       'shifting agriculture', 'water'], dtype=object)

In [99]:
gdf['driver'] = gdf['driver'].replace({
    'river, lake, ocean': 'other', 
    'other temporary crops': 'other',
    'aquaculture': 'other',
    'sugar cane': 'other',
    'beach, dune, snad': 'other',
    'rocky outcrop': 'other',
    'hypersaline tidal flat': 'other',
     'rice': 'other',
    'cotton': 'other',
    'citrus': 'other',
    'water': 'other',
})


In [ ]:
cls, count = np.unique(gdf.driver, return_counts=True)


#for i in range(len(cls)):
#    print(cls[i], count[i])

## Overlapping samples

- identify where examples from different datasets overlap with each other
- disregard those where the labels are the same - that's good news
- look at the rest

In [102]:
gdf.crs = 4326

gdf['geometry'] = gdf.to_crs(3857).geometry.buffer(500, cap_style=3).to_crs(4326)
# Perform spatial join to find intersecting polygons
# This creates a new GeoDataFrame where each row represents pairs of intersecting polygons
intersections = gpd.sjoin(gdf, gdf, how="inner", op='intersects')

# intersections will contain all pairs of polygons that intersect
# Note: this includes each polygon intersecting with itself, so you may want to remove those entries
intersections = intersections[intersections.index != intersections.index_right]
#intersections.to_file('../data/tmp/tmp.gpkg')

/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [103]:
disagreeing_overlaps = intersections.loc[~(intersections.driver_left == intersections.driver_right)]

In [140]:
gdf_without_conflicts = gdf[~gdf.index.isin(disagreeing_overlaps.index)].reset_index(drop=True)

In [137]:
len(disagreeing_overlaps.index.unique())

35150

In [104]:
clss, counts = np.unique(disagreeing_overlaps.driver_left, return_counts=True)

In [105]:
disagreeing_overlaps['class_mismatch'] = disagreeing_overlaps.driver_left + ", " + disagreeing_overlaps.driver_right
clss, counts = np.unique(disagreeing_overlaps.class_mismatch, return_counts=True)





/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
clss, counts = np.unique(disagreeing_overlaps.dataset_left, return_counts=True)
total_clss, total_counts = np.unique(gdf.dataset, return_counts=True)

clss = list(clss)
counts = list(counts)
clss.insert(3, "coffee_maskell.csv")
counts.insert(3, 0)
clss = np.array(clss)
counts = np.array(counts)


for _class, count, total_count in zip(clss, counts, total_counts):
    print(_class, " ", int(100*(count/total_count)), "% ")

## Write full dataset to disk

In [333]:
#Disregarding laos agriculture because it's not good

gdf_without_conflicts = gdf_without_conflicts.loc[gdf_without_conflicts.dataset != 'laos_shifting_agriculture.csv'].reset_index(drop=True)

In [ ]:
gdf_without_conflicts = gdf_without_conflicts.drop('split', axis=1)

In [334]:
pd.DataFrame(gdf_without_conflicts).to_csv('../data/labels/dataset_v4/labels.csv', index=False)

,geometry,driver,sampleid,dataset
0,"POLYGON ((97.10222 23.40982, 97.10222 23.40157...",rubber,20000000,rubber.csv
1,"POLYGON ((97.21218 23.38233, 97.21218 23.37408...",rubber,20000001,rubber.csv
2,"POLYGON ((97.01976 23.33652, 97.01976 23.32827...",rubber,20000002,rubber.csv
3,"POLYGON ((97.39543 23.32735, 97.39543 23.31910...",rubber,20000004,rubber.csv
4,"POLYGON ((97.01976 23.14410, 97.01976 23.13584...",rubber,20000005,rubber.csv
...,...,...,...,...
1253262,"POLYGON ((107.14650 20.97384, 107.14650 20.965...",mining,140033247,mines.csv
1253263,"POLYGON ((107.15567 20.97384, 107.15567 20.965...",mining,140033248,mines.csv
1253264,"POLYGON ((107.12818 20.96467, 107.12818 20.956...",mining,140033249,mines.csv
1253265,"POLYGON ((107.14650 20.96467, 107.14650 20.956...",mining,140033250,mines.csv


## Sampling limited number of examples per class

 - first, do it just by simply randomly sampling
 - look at the results in QGIS and check spatial bias (e.g., if 

In [ ]:
gdf = pd.read_csv('../data/labels/dataset_v4/labels.csv', index=False)

In [335]:
gdf = gdf_without_conflicts 

In [336]:
driver_classes = np.unique(gdf.driver, return_counts=True)[0]
max_count_per_class = 5000
def flatten(nested_list):
    return [x for xs in nested_list for x in xs]


indices = []
for _class in driver_classes:
    try:
        class_samples = gdf.loc[gdf.driver == _class].sample(max_count_per_class, random_state=1).index
    except ValueError:
        class_samples = gdf.loc[gdf.driver == _class].index
    indices.append(class_samples.tolist())

indices = flatten(indices)

subset = gdf.loc[indices]

In [337]:
cls, count = np.unique(subset.driver, return_counts=True)


for i in range(len(cls)):
    print(cls[i], count[i])

cocoa 5000
coffee 416
forest plantation 5000
forest regrowth 5000
mining 5000
other 5000
palm oil 5000
pasture 5000
rubber 5000
shifting agriculture 5000
soy 5000
urban 5000


In [338]:
subset['geometry'] = subset['geometry'].centroid

/tmp/ipykernel_1298769/595281034.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset['geometry'] = subset['geometry'].centroid


In [339]:
subset.to_file('../data/tmp/subset_5k_per_class.gpkg')
pd.DataFrame(subset).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

/tmp/ipykernel_1298769/595281034.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset['geometry'] = subset['geometry'].centroid


## Block data split

In [3]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])

In [4]:
gdf = gpd.GeoDataFrame(df)


In [17]:

coordinates = (gdf.geometry.x, gdf.geometry.y)


array([-107.73220307,  -32.66179794,  177.49976481,   28.01625029])

In [96]:
grid_geodf

,geometry
0,"POLYGON ((-106.73220 -32.66180, -106.73220 -31..."
1,"POLYGON ((-105.73220 -32.66180, -105.73220 -31..."
2,"POLYGON ((-104.73220 -32.66180, -104.73220 -31..."
3,"POLYGON ((-103.73220 -32.66180, -103.73220 -31..."
4,"POLYGON ((-102.73220 -32.66180, -102.73220 -31..."
...,...
17095,"POLYGON ((173.26780 26.33820, 173.26780 27.338..."
17096,"POLYGON ((174.26780 26.33820, 174.26780 27.338..."
17097,"POLYGON ((175.26780 26.33820, 175.26780 27.338..."
17098,"POLYGON ((176.26780 26.33820, 176.26780 27.338..."


In [40]:

out = vd.block_split(
    coordinates,
    spacing=1,
    #random_state=123,
)

In [41]:
blocks = np.unique(out[1])

In [90]:
len(blocks)

2026

In [83]:
#Find such split that the each fold has at least 35 examples of coffee

while True:
    # Shuffle the array
    np.random.shuffle(blocks)

    # Split the array into 5 equal parts
    split_arrays = np.array_split(blocks, 5)

    '''
    split 0-3 are for 4-fold cross validation (train-val)
    split 4 is test set 

    '''

    splits = []

    for item in out[1]:
        if item in split_arrays[0]:
            splits.append(0)
        elif item in split_arrays[1]:
            splits.append(1)
        elif item in split_arrays[2]:
            splits.append(2)
        elif item in split_arrays[3]:
            splits.append(3)
        elif item in split_arrays[4]:
            splits.append(4)


    splits = np.array(splits)
    gdf['split'] = splits
    subset = gdf

    
    min_coffee = 9999
    for i in range(5):
        classes, counts = np.unique(subset.loc[subset.split == i].driver, return_counts=True)
        itemindex = np.where(classes == 'coffee')
        if counts[itemindex] < min_coffee:
            min_coffee = counts[itemindex]
    print(min_coffee)
    if min_coffee > 35:
        break

[2]
[14]
[29]
[15]
[21]
[22]
[17]
[11]
[14]
[45]


In [63]:
gdf['block'] = out[1]

In [64]:
subset = gdf

In [84]:
classes, counts = np.unique(subset.loc[subset.split == 0].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1787  200  946 1171  820  942 1343 1277  735  766  663 1200]


In [85]:
classes, counts = np.unique(subset.loc[subset.split == 1].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1658   69  926  900 1384 1427 1124  831  879  993 1390 1213]


In [86]:
classes, counts = np.unique(subset.loc[subset.split == 2].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 225   45 1322 1009  584  968  926 1043  488 1112  852  914]


In [87]:
classes, counts = np.unique(subset.loc[subset.split == 3].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 132   53  768  917 1334 1018  817  867 1362 1044  972  693]


In [88]:
classes, counts = np.unique(subset.loc[subset.split == 4].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1196   49 1019  989  867  640  781  967 1528 1078 1115  965]


In [98]:
subset['split'].map(folds_to_split_mapping) 

0         test
1        train
2        train
3         test
4         test
         ...  
55298    train
55299    train
55300    train
55301    train
55302    train
Name: split, Length: 55303, dtype: object

In [89]:
#pd.DataFrame(subset).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)
subset.to_file('../data/tmp/labels_tmp2.gpkg')


In [35]:
subset

,geometry,driver,sampleid,dataset,split,block
0,POINT (-80.06997 -1.21407),cocoa,110000835,cocoa_ecuador.csv,1,8862
1,POINT (-3.25273 6.05541),cocoa,30005848,cocoa.csv,2,10934
2,POINT (-1.83249 6.04624),cocoa,30005875,cocoa.csv,0,10935
3,POINT (-6.44139 5.59727),cocoa,30007330,cocoa.csv,4,10931
4,POINT (-6.21232 5.62476),cocoa,30007224,cocoa.csv,4,10931
...,...,...,...,...,...,...
55298,POINT (76.66305 10.75844),urban,100002981,worldcover.csv,0,12439
55299,POINT (106.68138 -6.33444),urban,100009782,worldcover.csv,1,7624
55300,POINT (-35.32238 -5.74967),urban,10427227,mapbiomas.csv,0,7767
55301,POINT (101.54393 3.15495),urban,100004466,worldcover.csv,2,10469


## Implementation of the block split myself

- this could be better because i can make adjustments to optimize the class ratios

Steps
- create grid
- do an intersection with the points gdf
- only keep grid cells containing points
- split cells into 5 groups
    - this can be random or I can e.g. split the cells into high-density cells and low-density cells and take this into account


In [ ]:

import geopandas as gpd
from shapely.geometry import Polygon, box
from typing import Tuple

def create_grid(square: Polygon, width: float, height: float) -> gpd.GeoDataFrame:

    minx, miny, maxx, maxy = square.bounds

    ncols = int((maxx - minx) / width)
    nrows = int((maxy - miny) / height)
    
    grid_cells = []
    for i in range(nrows):
        for j in range(ncols):
            cell_minx = minx + j * width
            cell_miny = miny + i * height
            cell_maxx = cell_minx + width
            cell_maxy = cell_miny + height
            grid_cells.append(box(cell_minx, cell_miny, cell_maxx, cell_maxy))
    
    grid_gdf = gpd.GeoDataFrame(grid_cells, columns=['geometry'])
    return grid_gdf

square_polygon = box(*gdf.total_bounds)  
cell_width, cell_height = 1, 1

grid_geodf = create_grid(square_polygon, cell_width, cell_height)

## TMP

In [5]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])



In [4]:

folds_to_split_mapping = {
    0 : 'train',
    1 : 'train',
    2 : 'train',
    3 : 'val',
    4 : 'test'
}
df['split'] = df['split'].map(folds_to_split_mapping) 

In [5]:
df_v3 = pd.read_csv('../data/labels/dataset_v3/labels_forestloss_filtered.csv')


In [6]:
np.unique(df_v3.loc[df_v3.split == 'val'].driver, return_counts=True)

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
        'mining', 'palm oil', 'pasture', 'rubber', 'shifting agriculture',
        'soy', 'urban'], dtype=object),
 array([5000,  335, 4983, 4984, 4996, 4986, 4983, 4984, 3434, 4998, 2859]))

In [8]:
np.unique(df_v3.loc[df_v3.split == 'val'].driver, return_counts=True)

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
        'mining', 'palm oil', 'pasture', 'rubber', 'shifting agriculture',
        'soy', 'urban'], dtype=object),
 array([ 442,   28, 1028, 1227, 1986, 1303, 1129, 1486,  671, 1568,  996]))

In [7]:
np.unique(df.driver, return_counts=True)

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
        'mining', 'other', 'palm oil', 'pasture', 'rubber',
        'shifting agriculture', 'soy', 'urban'], dtype=object),
 array([4998,  416, 4981, 4986, 4989, 4995, 4991, 4985, 4992, 4993, 4992,
        4985]))

In [9]:
np.unique(df.loc[df.split == 'val'].driver, return_counts=True)

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
        'mining', 'other', 'palm oil', 'pasture', 'rubber',
        'shifting agriculture', 'soy', 'urban'], dtype=object),
 array([ 987,   53, 1457, 1047, 1173, 1101, 1157, 1055,  633, 1150,  628,
        1051]))

In [6]:
np.unique(df.loc[df.split == 2].driver, return_counts=True)

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
        'mining', 'other', 'palm oil', 'pasture', 'rubber',
        'shifting agriculture', 'soy', 'urban'], dtype=object),
 array([ 454,   35,  931, 1014,  725,  886,  847,  999, 1640,  795,  970,
         937]))

In [8]:
np.unique(df.loc[df.split.isin([0,1,3])].driver, return_counts=True)

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
        'mining', 'other', 'palm oil', 'pasture', 'rubber',
        'shifting agriculture', 'soy', 'urban'], dtype=object),
 array([3483,  306, 3213, 2910, 3115, 3347, 3080, 3094, 2572, 3220, 3258,
        3219]))

In [9]:
np.unique(df.loc[df.split == 4].driver, return_counts=True)

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
        'mining', 'other', 'palm oil', 'pasture', 'rubber',
        'shifting agriculture', 'soy', 'urban'], dtype=object),
 array([1061,   75,  837, 1062, 1149,  762, 1064,  892,  780,  978,  764,
         829]))

In [7]:
gdf.loc[gdf.split == 0].shape

(11648, 5)

(11492, 5)

### Add country

In [9]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df, geometry='geometry')
world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

/tmp/ipykernel_2969217/1252657791.py:4: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [11]:
gdf['country'] = gdf.geometry.intersects(world).name

/tmp/ipykernel_2969217/291178877.py:1: UserWarning: The indices of the two GeoSeries are different.
  gdf['country'] = gdf.geometry.intersects(world).name
/tmp/ipykernel_2969217/291178877.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  gdf['country'] = gdf.geometry.intersects(world).name


In [14]:
gdf.crs = 4326

In [29]:
gdf_with_countries = gdf.sjoin(world[['name', 'geometry']], how="left")


In [30]:
gdf_with_countries['country'] = gdf_with_countries.name

In [31]:
gdf_with_countries = gdf_with_countries.drop(['index_right', 'name'], axis=1)

In [38]:
pd.DataFrame(gdf_with_countries).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

In [39]:
gdf_with_countries

,geometry,driver,sampleid,dataset,split,country
0,POINT (-80.06997 -1.21407),cocoa,110000835,cocoa_ecuador.csv,1,Ecuador
1,POINT (-3.25273 6.05541),cocoa,30005848,cocoa.csv,2,Côte d'Ivoire
2,POINT (-1.83249 6.04624),cocoa,30005875,cocoa.csv,0,Ghana
3,POINT (-6.44139 5.59727),cocoa,30007330,cocoa.csv,3,Côte d'Ivoire
4,POINT (-6.21232 5.62476),cocoa,30007224,cocoa.csv,3,Côte d'Ivoire
...,...,...,...,...,...,...
55298,POINT (76.66305 10.75844),urban,100002981,worldcover.csv,1,India
55299,POINT (106.68138 -6.33444),urban,100009782,worldcover.csv,2,Indonesia
55300,POINT (-35.32238 -5.74967),urban,10427227,mapbiomas.csv,4,Brazil
55301,POINT (101.54393 3.15495),urban,100004466,worldcover.csv,2,Malaysia
